In [ ]:
import numpy as np

from monte_carlo import (
    MonteCarloEngine,
    HullWhiteProcess,
    HullWhiteTheta,
    BlackScholesProcess,
)



In [ ]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display


def plot_df_curve(alpha_r, sigma_r):
    # Market data / model parameters are assembled outside the engine.
    times = [0.0, 0.5, 1.0, 2.0, 5.0, 10.0]
    dfs = [1.0, 0.99, 0.975, 0.94, 0.85, 0.70]



    theta_model = HullWhiteTheta(times, dfs, alpha_r, sigma_r)
    rate = HullWhiteProcess(
        initial_value=theta_model.fwd_rate(0),
        alpha=alpha_r,
        sigma=sigma_r,
        theta=theta_model.theta
    )
    corr = np.array([
        [1.0]
    ])

    engine = MonteCarloEngine(
        processes={"rate": rate},
        correlation=corr,
        maturity=10.0,
        n_steps=250,
        n_paths=100_000,
        seed=45,
        discount_curve_key="rate",
    )

    result = engine.simulate()


    time_grid = result["time"]
    df_curve = result["discount_factors"].mean(axis=0)
    df_zcb = np.array([rate.zcb(0, theta_model.fwd_rate(0), t) for t in time_grid])
    df_curve_theoretical = theta_model.df(time_grid)

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=time_grid,
            y=df_curve,
            mode="lines",
            name="Discount Factor",
        )
    )

    fig.add_trace(
        go.Scatter(
            x=time_grid,
            y=df_curve_theoretical,
            mode="lines",
            name="Theoretical Discount Factor",
        )
    )

    fig.add_trace(
        go.Scatter(
            x=time_grid,
            y=df_zcb,
            mode="lines",
            name="ZCB Discount Factor",
        )
    )

    fig.update_layout(
        title=f"HW Discount Factor Curve (alpha={alpha_r:.3f}, sigma={sigma_r:.3f})",
        xaxis_title="Time (T)",
        yaxis_title="Discount Factor",
        yaxis=dict(range=[0, 1.05]),
        template="plotly_white",
    )

    fig.show()

alpha_r_slider = widgets.FloatSlider(
    value=0.3,
    min=0.00,
    max=0.9,
    step=0.005,
    description="alpha_r",
    continuous_update=False,
)

sigma_r_slider = widgets.FloatSlider(
    value=0.02,
    min=0.00,
    max=0.2,
    step=0.002,
    description="sigma_r",
    continuous_update=False,
)

ui = widgets.VBox([alpha_r_slider, sigma_r_slider])

out = widgets.interactive_output(
    plot_df_curve,
    {
        "alpha_r": alpha_r_slider,
        "sigma_r": sigma_r_slider,
    },
)

display(ui, out)

